# 07 - 自适应 RAG（查询复杂度路由）

**复杂度：** ⭐⭐⭐⭐

**应用场景：** 混合工作负载、成本优化、性能/质量平衡

**核心特性：** 对查询复杂度进行分类并路由到最优策略。

**路由逻辑：**
```
简单查询    → 快速相似度搜索
中等查询    → MMR（最大边界相关性）获取多样性
复杂查询    → HyDe（假设性文档嵌入）提升语义匹配
```

In [6]:
import sys
sys.path.append('../..')

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from shared.config import HF_VECTOR_STORE_PATH, DEFAULT_MODEL
from shared.utils import load_vector_store, print_section_header, format_docs
from shared.prompts import COMPLEXITY_CLASSIFIER_PROMPT, ADAPTIVE_RAG_PROMPT, HYDE_PROMPT
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

print_section_header("设置：自适应 RAG")

embeddings = create_embeddings()
vectorstore = load_vector_store(HF_VECTOR_STORE_PATH, embeddings)
llm = create_chat_model(model=DEFAULT_MODEL, temperature=0)

# 创建检索器
similarity_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5}
)

print("✅ 设置完成！")


设置：自适应 RAG



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\advanced_architectures\..\..\data\vector_stores\huggingface_embeddings
✅ 设置完成！


## 2. 复杂度分类器

In [7]:
print_section_header("复杂度分类器")

complexity_classifier = COMPLEXITY_CLASSIFIER_PROMPT | llm | StrOutputParser()

# 测试分类器
test_queries = [
    "What is FAISS?",  # SIMPLE
    "Compare OpenAI vs HuggingFace embeddings",  # MEDIUM
    "How to architect production RAG with privacy and cost constraints?"  # COMPLEX
]

for query in test_queries:
    complexity = complexity_classifier.invoke({"query": query}).strip()
    print(f"{complexity:8} | {query}")


复杂度分类器

SIMPLE   | What is FAISS?
MEDIUM   | Compare OpenAI vs HuggingFace embeddings
MEDIUM   | How to architect production RAG with privacy and cost constraints?


## 3. 自适应路由器

In [8]:
print_section_header("自适应路由器")

# 为复杂查询准备的 HyDe 生成器
hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def adaptive_route(query: str):
    """将查询路由到适当的检索策略。"""
    complexity = complexity_classifier.invoke({"query": query}).strip()
    
    if "SIMPLE" in complexity:
        docs = similarity_retriever.invoke(query)
        strategy = "简单-相似度"
    elif "MEDIUM" in complexity:
        docs = mmr_retriever.invoke(query)
        strategy = "中等-MMR"
    else:  # 复杂
        hypo_doc = hyde_generator.invoke({"question": query})
        docs = vectorstore.similarity_search(hypo_doc, k=4)
        strategy = "复杂-HyDe"
    
    return {"context": format_docs(docs), "input": query, "strategy": strategy}

print("✓ 自适应路由器已配置")
print("  - 简单 → 相似度搜索（快速）")
print("  - 中等 → MMR（多样化）")
print("  - 复杂 → HyDe（语义增强）")


自适应路由器

✓ 自适应路由器已配置
  - 简单 → 相似度搜索（快速）
  - 中等 → MMR（多样化）
  - 复杂 → HyDe（语义增强）


## 4. 自适应 RAG 链

In [9]:
print_section_header("自适应 RAG 链")

adaptive_chain = (
    RunnableLambda(adaptive_route)
    | ADAPTIVE_RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✓ 自适应 RAG 链已创建\n")

# 使用不同复杂度的查询进行测试
test_cases = [
    ("What is a retriever?", "SIMPLE"),
    ("Compare similarity vs MMR", "MEDIUM"),
    ("How to handle ambiguous queries in multi-domain systems?", "COMPLEX")
]

for query, expected in test_cases:
    print(f"\n查询（{expected}）：'{query}'")
    print("=" * 80)
    response = adaptive_chain.invoke(query)
    # 仅显示前 200 个字符
    print(response[:200] + "..." if len(response) > 200 else response)
    print()


自适应 RAG 链

✓ 自适应 RAG 链已创建


查询（SIMPLE）：'What is a retriever?'
Based on the provided context, a "retriever" is not explicitly defined as a standalone term. However, the context discusses **retrieval tools** that are used to fetch relevant documents from a source ...


查询（MEDIUM）：'Compare similarity vs MMR'
The provided context does not directly compare similarity search and MMR (Maximum Marginal Relevance). It discusses retrieval tools, indexing, vector stores, and semantic search, but MMR is not mentio...


查询（COMPLEX）：'How to handle ambiguous queries in multi-domain systems?'
Based on the context provided, handling ambiguous queries in multi-domain systems can be approached using the **MMR (Maximum Marginal Relevance)** strategy described. Specifically:

- **Contextual sea...



## 总结

**流程：**
```
查询 → 分类复杂度 → 路由到策略 → 检索 → LLM → 响应
```

**优势：**
✅ 优化成本（简单查询使用快速路径）  
✅ 平衡质量/速度  
✅ 适应查询难度  
✅ 可扩展以应对混合工作负载  

**局限性：**
- 分类开销
- 需要调整阈值
- 调试更复杂

**生产建议：**
- 缓存分类结果
- 监控路由分布
- A/B 测试路由逻辑
- 添加回退策略

**下一步：** [08_corrective_rag.ipynb](08_corrective_rag.ipynb) - CRAG 与网络搜索